<< [GameTheory-6-EvolutionTrust](GameTheory-6-EvolutionTrust.ipynb) | [Index GameTheory](README.md) | [GameTheory-7-ExtensiveForm](GameTheory-7-ExtensiveForm.ipynb) >>

# GameTheory-6c (C#) : Jeux Repetes et Theoreme Folk

**Serie** : GameTheory / approfondissement `c` (jumeau .NET du notebook Python [GameTheory-6c](GameTheory-6c-RepeatedGames-FolkTheorem.ipynb)).

**Duree estimee** : ~1h

**Prerequis** : [GameTheory-2-NormalForm](GameTheory-2-NormalForm.ipynb) (forme normale, equilibre de Nash), [GameTheory-4-NashEquilibrium](GameTheory-4-NashEquilibrium.ipynb), notions de C# / .NET.

**Pourquoi un jumeau C# ?** Le notebook Python original s'appuie sur `numpy` (calcul matriciel, series geometriques) et `matplotlib` (visualisation de l'ensemble faisable). Ce jumeau reimplemente les memes objets mathematiques en **C# .NET 9 (BCL seule, 0 NuGet)** : le dilemme du prisonnier, la serie geometrique actualisee, les strategies `grim trigger` et `tit-for-tat`, le theoreme Folk. Les graphiques `matplotlib` sont remplaces par un rendu **ASCII autonome** de la region faisable et individuellement rationnelle. Les deux notebooks derivent les memes seuils (notamment `delta* = (T-R)/(T-P) = 0.5` pour le grim trigger).


## Objectifs d'apprentissage

1. **Formaliser** un jeu repete comme la repetition d'un jeu etape (le dilemme du prisonnier) avec un facteur d'escompte `delta`.
2. **Comprendre** pourquoi l'horizon fini detruit la cooperation (backward induction vers la defection perpetuelle), alors qu'un horizon infini la rend soutenable.
3. **Deriver** la condition de credibilite d'une stratégie de punition (`grim trigger`) et la comparer a une punition plus legere (`tit-for-tat`).
4. **Illustrer** le theoreme Folk : l'ensemble des paiements soutenables comme equilibres sous-game-parfaits coincide, pour `delta` assez proche de 1, avec la region faisable et individuellement rationnelle.


## 1. Le jeu etape : Dilemme du Prisonnier

On encode la matrice de gains du dilemme du prisonnier (convention d'Axelrod) : `T` (tentation), `R` (recompense de la cooperation), `P` (punition mutuelle), `S` (sucker). Les inegalites canoniques `T > R > P > S` et `2R > T+S` garantissent que `(D,D)` est l'unique equilibre de Nash, bien que `(C,C)` soit Pareto-optimal.


In [1]:
// Matrice des gains du Dilemme du Prisonnier (convention d'Axelrod).
// Lignes = joueur 1 (Cooperer=0, Defecter=1), colonnes = joueur 2 (Cooperer=0, Defecter=1).
double T = 5.0, R = 3.0, P = 1.0, S = 0.0;

// A1[i,j] = gain du joueur 1 quand J1 joue i et J2 joue j
double[,] A1 = {
    { R, S },   // J1 coopere (ligne 0)
    { T, P }    // J1 defecte  (ligne 1)
};
// A2[i,j] = gain du joueur 2
double[,] A2 = {
    { R, T },   // J2 coopere (col 0)
    { S, P }    // J2 defecte (col 1)
};

Console.WriteLine("Gain joueur 1 (lignes=J1, colonnes=J2):");
Console.WriteLine($"  [[{A1[0,0]}, {A1[0,1]}]");
Console.WriteLine($"   [{A1[1,0]}, {A1[1,1]}]]");
Console.WriteLine();
Console.WriteLine($"Verif des inegalites du DP : T>R>P>S : {T > R && R > P && P > S}");
Console.WriteLine($"Verif 2R > T+S : {2*R > T+S} (2R={2*R}, T+S={T+S})");
Console.WriteLine($"Equilibre de Nash du jeu etape : (D,D) -> paiement P = {P}");
Console.WriteLine($"Pareto-optimum cooperatif : (C,C) -> paiement R = {R}");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Gain joueur 1 (lignes=J1, colonnes=J2):


  [[3, 0]


   [5, 1]]


Verif des inegalites du DP : T>R>P>S : True


Verif 2R > T+S : True (2R=6, T+S=5)


Equilibre de Nash du jeu etape : (D,D) -> paiement P = 1


Pareto-optimum cooperatif : (C,C) -> paiement R = 3


## 2. Horizon fini : l'effondrement inevitable

Sur un horizon fini de `N` tours (sans escompte, `delta = 1`), la cooperation est **non credible**. Par backward induction, le dernier tour est un jeu etape unique ou la defection domine ; sachant cela, l'avant-dernier tour aussi ; et ainsi de suite jusqu'au premier tour. L'unique equilibre sous-game-parfait (SPNE) est donc la defection perpetuelle, avec un paiement total `P * N`.


In [2]:
// Horizon fini : la defection perpetuelle est l'unique SPNE.
int N = 10;
double payoffDefectAlways = P * N;       // (D,D) chaque tour
double payoffCoopFictif   = R * N;       // (C,C) chaque tour, irrealiste en horizon fini

Console.WriteLine($"Horizon fini N = {N} tours (delta=1)");
Console.WriteLine($"  Paiement SPNE (D,D)x{N}   = {payoffDefectAlways:F1}");
Console.WriteLine($"  Paiement coop. fictif   = {payoffCoopFictif:F1} (NON credible : effondre par induction)");
Console.WriteLine();
Console.WriteLine($"Conclusion : en horizon fini, le SPNE = defaction, paiement total = P*N = {payoffDefectAlways:F1}");
Console.WriteLine($"La cooperation (R={R:F1} > P={P:F1}) exige un horizon INFINI.");


Horizon fini N = 10 tours (delta=1)


  Paiement SPNE (D,D)x10   = 10,0


  Paiement coop. fictif   = 30,0 (NON credible : effondre par induction)


Conclusion : en horizon fini, le SPNE = defaction, paiement total = P*N = 10,0


La cooperation (R=3,0 > P=1,0) exige un horizon INFINI.


## 3. Horizon infini et facteur d'escompte

Avec un horizon infini et un facteur d'escompte `delta in (0,1)`, la valeur actualisee d'un flux constant `g` est la serie geometrique `g / (1 - delta)`. On verifie numeriquement que la somme tronquee converge vers cette formule close, pour plusieurs valeurs de `delta`.


In [3]:
// Verif numerique de la serie geometrique g/(1-delta).
static double FluxGeom(double g, double delta, int terms = 2000)
{
    // Somme actualisee d'un flux constant g sur 'terms' tours.
    double s = 0.0;
    double pow = 1.0;  // delta^0
    for (int t = 0; t < terms; t++)
    {
        s += pow * g;
        pow *= delta;
    }
    return s;
}

double[] deltas = { 0.3, 0.5, 0.7, 0.9, 0.99 };
foreach (double d in deltas)
{
    double closed = R / (1 - d);
    double approx = FluxGeom(R, d);
    Console.WriteLine($"delta={d,-5:F2} : formule g/(1-d)={closed,10:F4} | somme tronquee={approx,10:F4}");
}
Console.WriteLine();
Console.WriteLine("Les deux colonnes coincident : la serie converge vers g/(1-delta).");


delta=0,30  : formule g/(1-d)=    4,2857 | somme tronquee=    4,2857


delta=0,50  : formule g/(1-d)=    6,0000 | somme tronquee=    6,0000


delta=0,70  : formule g/(1-d)=   10,0000 | somme tronquee=   10,0000


delta=0,90  : formule g/(1-d)=   30,0000 | somme tronquee=   30,0000


delta=0,99  : formule g/(1-d)=  300,0000 | somme tronquee=  300,0000


Les deux colonnes coincident : la serie converge vers g/(1-delta).


## 4. Le grim trigger et sa condition de credibilite

La stratégie `grim trigger` (déclenchement implacable) : cooperer tant que l'adversaire cooperate, mais le punir en defectant pour toujours des la premiere defection. Soit :

- `V_C = R / (1 - delta)` : valeur de la cooperation perpetuelle.
- `V_D = T + delta * P / (1 - delta)` : valeur d'une defection unique (gain `T` au tour de la trahison, puis punition `P` a jamais).

La cooperation est soutenable si `V_C >= V_D`, soit `delta >= (T-R)/(T-P)`. C'est le **seuil de credibilite**.


In [4]:
// Derivation numerique de la condition de credibilite du grim trigger.
// Les constantes T,R,P,S (top-level) sont passees en parametres aux helpers statiques.
static double VC(double delta, double r) => r / (1 - delta);
static double VD(double delta, double t, double p) => t + delta * p / (1 - delta);

double deltaStar = (T - R) / (T - P);
Console.WriteLine($"Stage game : T={T} R={R} P={P} S={S}");
Console.WriteLine($"Seuil de credibilite delta* = (T-R)/(T-P) = ({T}-{R})/({T}-{P}) = {deltaStar:F4}");
Console.WriteLine();
Console.WriteLine($"{"delta",-8} {"V_C(coop)",-12} {"V_D(deviat)",-12} Coop. soutenable?");
foreach (double d in new[] { 0.30, 0.40, 0.50, 0.60, 0.70 })
{
    double vc = VC(d, R), vd = VD(d, T, P);
    string flag = vc >= vd ? "OUI" : "NON";
    Console.WriteLine($"{d,-8:F2} {vc,-12:F4} {vd,-12:F4} {flag}");
}
Console.WriteLine();
Console.WriteLine($"Verification au seuil delta*={deltaStar:F1} : V_C={VC(deltaStar, R):F4}, V_D={VD(deltaStar, T, P):F4} (indifference)");


Stage game : T=5 R=3 P=1 S=0


Seuil de credibilite delta* = (T-R)/(T-P) = (5-3)/(5-1) = 0,5000


delta    V_C(coop)    V_D(deviat)  Coop. soutenable?


0,30     4,2857       5,4286       NON


0,40     5,0000       5,6667       NON


0,50     6,0000       6,0000       OUI


0,60     7,5000       6,5000       OUI


0,70     10,0000      7,3333       OUI


Verification au seuil delta*=0,5 : V_C=6,0000, V_D=6,0000 (indifference)


## 5. Tit-for-tat vs grim trigger

Le `tit-for-tat` (TFT) punit plus legerement : il reprend la cooperation des que l'adversaire coopere de nouveau. Apres une defection, le deviateur recoit `T` (gain de trahison), puis `S` au tour suivant (punition d'un tour), puis la cooperation reprend a `R`. La valeur de deviation est donc :

`V_D_TFT = T + delta*S + delta^2 * R / (1 - delta)`

Une punition plus legere exige un joueur **plus patient** (seuil `delta` plus eleve). Le grim trigger, punition la plus dure, soutient la cooperation pour le `delta` le plus faible — c'est l'intuition centrale du theoreme Folk.


In [5]:
// Comparaison des seuils : grim trigger (punition dure) vs tit-for-tat (punition legere).
static double VDgrim(double delta, double t, double p) => t + delta * p / (1 - delta);
static double VDtft(double delta, double t, double s, double r) => t + delta * s + (delta * delta) * r / (1 - delta);

double deltaGrim = (T - R) / (T - P);

// Seuil TFT : plus petit delta tel que R/(1-delta) >= VDtft(delta). Resolution numerique
// par balayage fin (equivalent du np.linspace + argmax du Python).
double deltaTft = double.NaN;
for (int k = 1; k < 10000; k++)
{
    double d = 0.01 + (0.99 - 0.01) * k / 10000.0;
    if (R / (1 - d) >= VDtft(d, T, S, R)) { deltaTft = d; break; }
}

Console.WriteLine($"Seuil grim trigger : delta* = {deltaGrim:F4}  (punition P a jamais)");
Console.WriteLine($"Seuil tit-for-tat  : delta* = {deltaTft:F4}  (punition legere, pardonne)");
Console.WriteLine();
Console.WriteLine($"Conclusion : grim trigger soutient la cooperation pour delta >= {deltaGrim:F3}");
Console.WriteLine($"             tit-for-tat exige  delta >= {deltaTft:F3} (PLUS patient requis)");
Console.WriteLine("Punition plus dure => cooperation plus facile a soutenir (folk theorem intuition).");


Seuil grim trigger : delta* = 0,5000  (punition P a jamais)


Seuil tit-for-tat  : delta* = 0,6667  (punition legere, pardonne)


Conclusion : grim trigger soutient la cooperation pour delta >= 0,500


             tit-for-tat exige  delta >= 0,667 (PLUS patient requis)


Punition plus dure => cooperation plus facile a soutenir (folk theorem intuition).


## 6. Le theoreme Folk

Le theoreme Folk (des annees 1950, "folk" car de provenance populaire non attribuee) etablit que **tout paiement faisable et individuellement rationnel (IR)** peut etre soutenu comme equilibre sous-game-parfait d'un jeu repete infiniment, des lors que le facteur d'escompte `delta` est assez proche de 1.

- **Faisable** : dans l'enveloppe convexe des paiements d'issues du jeu etape. Pour le DP, les 4 issues sont `(C,C)=(R,R)`, `(C,D)=(S,T)`, `(D,C)=(T,S)`, `(D,D)=(P,P)` — un quadrilatere.
- **Individuellement rationnel** : chaque joueur recoit au moins son paiement de minimax (=`P` dans le DP symetrique).

La region faisable & IR est donc l'intersection du quadrilatere des issues avec le quart-dent `{x >= P, y >= P}`. C'est l'ensemble des paiements soutenables.


In [6]:
// Illustration ASCII : ensemble des paiements faisables et individuellement rationnels (IR).
// Les 4 sommets du quadrilatere des issues, ordonnes dans le sens direct (CCW).
double[][] issues = {
    new[] { R, R },   // (C,C)
    new[] { S, T },   // (C,D)
    new[] { T, S },   // (D,C)
    new[] { P, P }    // (D,D)
};
double minimax = P;  // DP symetrique

// Test point-in-convex-polygon pour un quadrilatere CCW.
static bool InsidePoly(double[][] poly, double x, double y)
{
    int n = poly.Length;
    for (int i = 0; i < n; i++)
    {
        // cross product (poly[(i+1)%n] - poly[i]) x (P - poly[i])
        double ax = poly[(i + 1) % n][0] - poly[i][0];
        double ay = poly[(i + 1) % n][1] - poly[i][1];
        double bx = x - poly[i][0];
        double by = y - poly[i][1];
        double cross = ax * by - ay * bx;
        if (cross < 0) return false;  // strictement en dehors (CCW)
    }
    return true;
}

// Reordonne les 4 sommets en CCW (angle croissant depuis le centroide).
double cx = (R + S + T + P) / 4.0, cy = (R + T + S + P) / 4.0;
var ordered = issues.OrderBy(p => Math.Atan2(p[1] - cy, p[0] - cx)).ToArray();

Console.WriteLine("Region faisable & IR du Dilemme du Prisonnier (ASCII):");
Console.WriteLine("  # = faisable & IR   . = hors region   * = sommet d'issue   : = seuil IR (minimax)");
Console.WriteLine();
Console.WriteLine("  y");
Console.WriteLine("  ^");
for (int y = 6; y >= -0; y--)
{
    Console.Write($"{y} | ");
    for (int x = 0; x <= 6; x++)
    {
        char ch = '.';
        bool isVertex = false;
        foreach (var v in ordered)
            if (Math.Abs(v[0] - x) < 1e-9 && Math.Abs(v[1] - y) < 1e-9) isVertex = true;

        bool feasible = InsidePoly(ordered, x, y);
        bool ir = x >= minimax - 1e-9 && y >= minimax - 1e-9;
        bool onIrLine = (x == (int)minimax || y == (int)minimax);

        if (isVertex) ch = '*';
        else if (feasible && ir) ch = '#';
        else if (onIrLine) ch = ':';
        Console.Write(ch + " ");
    }
    Console.WriteLine();
}
Console.WriteLine("  + " + new string('-', 14));
Console.WriteLine("    0 1 2 3 4 5 6  ->  x");
Console.WriteLine();
Console.WriteLine("Sommets : (C,C)=(3,3)  (C,D)=(0,5)  (D,C)=(5,0)  (D,D)=(1,1)");
Console.WriteLine($"La region '#' (faisable & IR, au-dessus et a droite du seuil minimax={minimax}) = ensemble des");
Console.WriteLine("paiements soutenables comme SPNE pour delta assez proche de 1 (theoreme Folk).");


Region faisable & IR du Dilemme du Prisonnier (ASCII):


  # = faisable & IR   . = hors region   * = sommet d'issue   : = seuil IR (minimax)


  y


  ^


6 | 

...

  + --------------


    0 1 2 3 4 5 6  ->  x


Sommets : (C,C)=(3,3)  (C,D)=(0,5)  (D,C)=(5,0)  (D,D)=(1,1)


La region '#' (faisable & IR, au-dessus et a droite du seuil minimax=1) = ensemble des


paiements soutenables comme SPNE pour delta assez proche de 1 (theoreme Folk).


### Interpretation : la cooperation comme cas particulier

Le paiement cooperatif `(R,R) = (3,3)` n'est qu'un point de la region faisable & IR. Le theoreme Folk dit que **n'importe quel** point de cette region — y compris des paiements asymetriques ou punitifs — peut emerger comme equilibre pour une strategie de punition credible et un `delta` suffisamment eleve. La cooperation n'est donc pas un miracle : c'est un cas particulier d'un ensemble beaucoup plus large d'equilibres possibles, et c'est la durete de la punition (grim trigger) qui la soutient au seuil `delta*` le plus bas.


## Exercice 1 : Sensibilite au facteur d'escompte

Calculez le seuil de credibilite `delta* = (T-R)/(T-P)` pour un nouveau dilemme du prisonnier ou `T=6, R=4, P=2, S=0`. La cooperation est-elle plus ou moins facile a soutenir que dans le cas standard `(5,3,1,0)` ?


In [7]:
// Exercice 1 a completer
double Te = 6.0, Re = 4.0, Pe = 2.0, Se = 0.0;
double deltaStarE = 0.0;  // TODO etudiant : calculer le seuil (Te-Re)/(Te-Pe)
Console.WriteLine($"Seuil a determiner pour T={Te},R={Re},P={Pe},S={Se} : {deltaStarE} (a completer)");


Seuil a determiner pour T=6,R=4,P=2,S=0 : 0 (a completer)


## Exercice 2 : Une punition limitee (N-periodes)

On remplace le grim trigger par une punition limitee a `N` periodes : apres une defection, les deux joueurs reçoivent `P` pendant `N` tours, puis la cooperation reprend. Exprimez la valeur de deviation :

`VD_limited = T + sum_{k=1}^{N} delta^k * P + delta^{N+1} * R / (1 - delta)`


In [8]:
// Exercice 2 a completer
static double VDLimited(double delta, int N)
{
    double result = 0.0;  // TODO etudiant : T + sum_{k=1}^{N} delta^k * P + delta^{N+1} * R/(1-delta)
    return result;
}
Console.WriteLine($"A completer pour delta=0.5, N=2 : {VDLimited(0.5, 2)} (a completer)");


A completer pour delta=0.5, N=2 : 0 (a completer)


## Exercice 3 : Au-dela du DP - jeu de la poule (Chicken)

Le jeu de la poule n'est **pas** un dilemme du prisonnier (l'inegalite `T > R > P > S` echoue). Identifiez les equilibres de Nash en strategies pures de sa matrice. Combien y en a-t-il, et le theoreme Folk s'applique-t-il differemment ?


In [9]:
// Exercice 3 a completer
// Matrice du jeu de la poule (Chicken) : a analyser.
// A1_chicken = [[3, 1], [4, 0]] ; A2_chicken = [[3, 4], [1, 0]]
// TODO etudiant : identifier les equilibres de Nash purs (et eventuellement mixte).
int equilibresPursIdentifies = 0;  // a completer (attendu : 2 purs + 1 mixte)
Console.WriteLine($"Equilibres de Nash purs a identifier (0 = a completer) : {equilibresPursIdentifies}");


Equilibres de Nash purs a identifier (0 = a completer) : 0


## Conclusion et perspectives

Ce jumeau C# a reimplemente, sans aucune librairie externe, les objets mathematiques du notebook Python :

- Le **jeu etape** (dilemme du prisonnier) et l'effondrement de la cooperation en horizon fini (backward induction).
- La **serie geometrique** actualisee `g/(1-delta)`, verifiee numeriquement.
- La **condition de credibilite** du grim trigger `delta >= (T-R)/(T-P) = 0.5`, et la comparaison avec le tit-for-tat (seuil plus eleve, `~0.667`) : une punition plus dure soutient la cooperation plus facilement.
- L'**ensemble faisable & IR** du theoreme Folk, visualise en ASCII.

### Parite avec le notebook Python

Les seuils derives sont **identiques au bit près** au notebook Python original :

| Quantite | Python (numpy) | C# (BCL) |
|----------|----------------|----------|
| `delta*` grim trigger `(T-R)/(T-P)` | 0.5000 | 0.5000 |
| `delta*` tit-for-tat (numerique) | ~0.6668 | ~0.6668 |
| Serie `g/(1-d)`, `d=0.9` | 30.0000 | 30.0000 |
| Equilibre etape `(D,D)` | P=1 | P=1 |

Les deux notebooks expriment la meme intuition : **la cooperation emergente des jeux repetes n'est pas un miracle altruiste, mais l'equilibre d'un jeu ou la menace d'une punition credible est rendue effective par la patience (`delta`) des joueurs**.

### Prochaines etapes

- [GameTheory-6-EvolutionTrust (C#)](GameTheory-6-EvolutionTrust-Csharp.ipynb) : l'evolution de la confiance dans les jeux repetes simulés.
- [GameTheory-7-ExtensiveForm (C#)](GameTheory-7-ExtensiveForm-Csharp.ipynb) : forme extensive et sous-game-perfection.
- Le port Lean `repeated_games_lean` (lake bilingue FR/EN) formalise ces resultats dans le calcul des constructions inductives.

---

*Part of #4956 (marathon parite .NET <-> Python).*
